# 1 · Full Research Cycle

A complete pass: **data → indicators → strategy/rules → SL·TP config → simulate → stats →
plots → interactive chart → save**. Run top-to-bottom; edit the config cells to taste.


## 0 · Start / setup


In [1]:
import sys, os, json, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..'))
    import quant

from pathlib import Path
from quant.data import get_ohlcv
from quant.indicators import add_emas, add_rsi, add_macd
from quant.engine import BacktestConfig, TakeProfit, Signals, run_backtest
from quant.strategies import EmaRibbon
from quant import signals as S
from quant import reporting as R, analytics as A
from quant.viz import (ResearchChart, equity_chart, equity_and_drawdown,
                       monthly_returns_heatmap, hour_weekday_heatmap)
print('quant', quant.__version__)

quant 0.1.0


## 1 · Select ticker & data source
`source='binance'` (PAXGUSDT/XAUTUSDT/BTCUSDT…) or `source='dukascopy'` (true spot **XAUUSD**,
needs `pip install dukascopy-python`).


In [2]:
SYMBOL = 'PAXGUSDT'      # e.g. 'XAUUSD' with source='dukascopy'
SOURCE = 'binance'       # 'binance' | 'dukascopy'
MARKET = 'spot'

## 2 · Timeframe & date range
Only this range is pulled; missing candles download incrementally, then cache.


In [6]:
TF   = '15m'
START, END = '2025-01-01', '2026-07-23'
TZ   = 'UTC'            # display tz for charts & session/time filters

df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, market=MARKET, tz=TZ)
print(f'{len(df):,} bars  {df.open_time.min()} -> {df.open_time.max()}')
df.tail(3)

[05:29:58] INFO | quant.data | cache EXTEND binance PAXGUSDT 15m -> incremental fetch
[05:29:58] INFO | 01. Parameters: market=spot, symbol=PAXGUSDT, interval=15m, start=2025-01-01 00:00:00+00:00, end=2026-07-23 00:00:00+00:00
[05:29:58] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[05:29:58] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data\binance_spot_PAXGUSDT_15m.parquet
[05:29:58] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data\.partials\binance_spot_PAXGUSDT_15m
[05:29:58] INFO | 05. Cache: MISS (no main cache or partial checkpoints found).
[05:29:58] INFO | 06. Fetch plan 1/1: 2025-01-01 00:00:00+00:00 -> 2026-07-23 00:00:00+00:00
[05:29:58] INFO | 07. Starting fetch range 1/1.


PAXGUSDT 15m:   0%|          | 0/55 [00:00<?, ?page/s]

[05:32:04] INFO | 08. Completed fetch range 1/1 | rows=54,435 | total_rows_fetched=54,435 | requests=55 | retries=0 | elapsed=126.3s
[05:32:05] INFO | 09. Cache consolidated and saved (parquet) | rows=54,435 | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:30:00+00:00 | cleared_partial_files=55
[05:32:05] INFO | 10. Fetch summary | new_rows=54,435 | successful_pages=55 | requests=55 | retries=0 | partial_saves=55 | elapsed=127.0s
[05:32:05] INFO | 11. Returning slice | rows=54,435 | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:30:00+00:00


54,435 bars  2025-01-01 00:00:00+00:00 -> 2026-07-22 00:30:00+00:00


,open_time,open,high,low,close,volume,quote_volume,num_trades,taker_buy_base,taker_buy_quote,t
54432,2026-07-22 00:00:00+00:00,4074.13,4085.00,4071.38,4084.99,23.5149,95920.106853,369,17.2440,70333.521245,2026-07-22 00:00:00+00:00
54433,2026-07-22 00:15:00+00:00,4084.99,4091.01,4084.99,4090.30,48.8087,199489.415380,340,32.4330,132525.021540,2026-07-22 00:15:00+00:00
54434,2026-07-22 00:30:00+00:00,4090.59,4093.80,4086.36,4088.27,12.2687,50187.226015,141,2.0796,8507.722088,2026-07-22 00:30:00+00:00


## 3 · Indicators
Add any indicator columns you want available to rules/plots (vectorized; safe to add many).


In [ ]:
df = add_emas(df, [20, 50, 100, 200])   # ema_20, ema_50, ema_100, ema_200
df = add_rsi(df, 14)                     # rsi_14
[c for c in df.columns if c.startswith(('ema_', 'rsi'))]

## 4 · Strategy — pick ONE of the two styles below


### 4a · Built-in strategy (one line)
Buy when price crosses EMA(20) **and** stays above EMA(50) for 3 candles; exit on the reverse cross.


In [ ]:
strategy = EmaRibbon(fast=20, slow=50, confirm_n=3)

### 4b · OR define your own buy/sell open & close rules inline
Compose vectorized primitives into entry/exit boolean arrays. This is the raw 'rules' interface.
(Run this cell to override `strategy` with a custom-signals runner; skip it to use 4a.)


In [ ]:
# BUY (open long)  : close crosses above EMA20  AND close has been above EMA50 for 3 bars
# SELL (close long): close crosses below EMA20   (or RSI overbought)
open_long  = S.all_of(S.cross_up(df, 'close', 'ema_200'),
                      S.last_all_above(df, 'close', 'ema_50', 2))
close_long = S.any_of(S.cross_down(df, 'close', 'ema_50'))
custom_signals = Signals(entry_long=open_long, exit_long=close_long)
print('buy signals:', int(open_long.sum()), '| sell signals:', int(close_long.sum()))

## 5 · Backtest config — SL / TP / costs / sizing
See `BacktestConfig` docstring for every field. Example: 0.6%% stop, laddered TP (50%% at 1R then
the rest at 2R with breakeven), risk 1%% of equity per trade, gold spread + commission.


In [ ]:
cfg = BacktestConfig(
    initial_cash=10_000,
    fee_bps=0, commission_per_lot=0.0, spread=0.20,   # Exness-style: spread + (usually 0) commission
    slippage_bps=1.0,
    exit_enabled=True,
    sl_mode='entry_pct', sl_value=0.6,                # or sl_mode='ref_col', sl_ref_long_col='swing_last_low'
    take_profits=(TakeProfit('rr', 1.0, close_pct=50, move_stop_mode='breakeven'),
                  TakeProfit('rr', 2.0, close_pct=100)),
    sizing_mode='risk_pct_equity', sizing_value=1.0,
    # leverage (optional): margin_enabled=True, leverage=500, contract_size=100, sizing_mode='lots', sizing_value=0.1,
)

## 6 · Run the simulation & see stats


In [ ]:
USE_CUSTOM_RULES = False   # True -> use the 4b custom signals; False -> use the 4a strategy
if USE_CUSTOM_RULES:
    res = run_backtest(df, custom_signals, cfg)
else:
    res = strategy.backtest(df, cfg)
R.print_summary(res, df=df)
res.stats

## 7 · Stat plots


In [ ]:
equity_and_drawdown(res.equity_curve)

In [ ]:
monthly_returns_heatmap(A.monthly_returns(res.equity_curve))

In [ ]:
hour_weekday_heatmap(res.trades, metric='total_pnl')   # when does it make/lose money?

## 8 · Interactive price + indicator chart with trades
Smooth at millions of candles. **Click legend entries to toggle** any line / markers / candles.
Pan & zoom reloads higher resolution for the visible window only.


In [ ]:
ch = ResearchChart(df, candles=True)
ch.add_ema(20); ch.add_ema(50); ch.add_ema(200)
ch.add_trades(res.trades)
ch.show()

## 9 · Save results
Writes trades, equity, stats, and a standalone HTML report to `reports/<run>/`.


In [ ]:
run_name = f'{SYMBOL}_{TF}_{START}_{END}'
out = Path('reports') / run_name; out.mkdir(parents=True, exist_ok=True)
res.trades.to_csv(out/'trades.csv', index=False)
res.equity_curve.to_csv(out/'equity.csv', index=False)
json.dump(res.stats, open(out/'stats.json','w'), indent=2, default=float)
R.to_html(res, str(out/'report.html'), df=df, price_df=df, title=run_name)
print('saved ->', out.resolve())